# Divisão Temporal: Treino e Out-of-Time (OOT)

Neste notebook, vamos realizar a divisão temporal dos dados de transações em:
- **Dataset de Treino (df_treino)**: Usado para treinar o modelo de detecção de lavagem de dinheiro
- **Dataset Out-of-Time (df_oot)**: Usado para validação temporal e cálculo de métricas

## 1. Importação de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 2. Caminho do Arquivo

In [2]:
file_path = r'C:\Users\win\Desktop\TCC\money_laundering\data\raw\trans_enriched.csv'

## 3. Leitura dos Dados

In [3]:
# Leitura do arquivo CSV
print(f"Carregando dados de: {file_path}")
df = pd.read_csv(file_path)

# Converter a coluna Timestamp para datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"\nDados carregados com sucesso!")
print(f"Total de registros: {len(df):,}")
print(f"\nPeríodo dos dados:")
print(f"Data inicial: {df['Timestamp'].min()}")
print(f"Data final: {df['Timestamp'].max()}")
print(f"\nPrimeiras linhas:")
df.head()

Carregando dados de: C:\Users\win\Desktop\TCC\money_laundering\data\raw\trans_enriched.csv

Dados carregados com sucesso!
Total de registros: 31,251,483

Período dos dados:
Data inicial: 2022-09-01 00:00:00
Data final: 2022-09-27 14:58:00

Primeiras linhas:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022-09-01 00:15:00,20,800104D70,20,800104D70,8095.07,US Dollar,8095.07,US Dollar,Reinvestment,...,Regents Credit Union,20,800104D70,2AA23697070,Sole Proprietorship #1,Regents Credit Union,20,800104D70,2AA23697070,Sole Proprietorship #1
1,2022-09-01 00:18:00,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,...,Bank of Philadelphia,3196,800107150,2AA23466990,Partnership #1,Bank of Philadelphia,3196,800107150,2AA23466990,Partnership #1
2,2022-09-01 00:23:00,1208,80010E430,1208,80010E430,2654.22,US Dollar,2654.22,US Dollar,Reinvestment,...,Bank of Danbury,1208,80010E430,2AA237140E0,Sole Proprietorship #2,Bank of Danbury,1208,80010E430,2AA237140E0,Sole Proprietorship #2
3,2022-09-01 00:19:00,3203,80010EA80,3203,80010EA80,13284.41,US Dollar,13284.41,US Dollar,Reinvestment,...,Savings Bank of Augusta,3203,80010EA80,2AA23134470,Sole Proprietorship #3,Savings Bank of Augusta,3203,80010EA80,2AA23134470,Sole Proprietorship #3
4,2022-09-01 00:27:00,20,800104D20,20,800104D20,9.72,US Dollar,9.72,US Dollar,Reinvestment,...,Regents Credit Union,20,800104D20,2AA1FB24CC0,Individual #1,Regents Credit Union,20,800104D20,2AA1FB24CC0,Individual #1


## 4. Análise da Distribuição Temporal

In [4]:
# Análise da distribuição por mês
df['Year_Month'] = df['Timestamp'].dt.to_period('M')
distribuicao_mensal = df.groupby('Year_Month').agg({
    'Timestamp': 'count',
    'Is Laundering': 'sum'
}).rename(columns={'Timestamp': 'Total_Transacoes', 'Is Laundering': 'Total_Lavagem'})

distribuicao_mensal['Percentual_Lavagem'] = (
    distribuicao_mensal['Total_Lavagem'] / distribuicao_mensal['Total_Transacoes'] * 100
).round(2)

print("Distribuição mensal dos dados:")
print(distribuicao_mensal)

Distribuição mensal dos dados:
            Total_Transacoes  Total_Lavagem  Percentual_Lavagem
Year_Month                                                     
2022-09             31251483          16041                0.05


## 5. Divisão Temporal: Treino e Out-of-Time

Vamos dividir os dados utilizando uma proporção de **80% para treino** e **20% para out-of-time**. 
A divisão será feita com base na ordem temporal das transações.

In [5]:
# Ordenar os dados por timestamp
df_sorted = df.sort_values('Timestamp').reset_index(drop=True)

# Definir o ponto de corte temporal (80% para treino, 20% para OOT)
split_ratio = 0.8
split_index = int(len(df_sorted) * split_ratio)

# Realizar a divisão
df_treino = df_sorted.iloc[:split_index].copy()
df_oot = df_sorted.iloc[split_index:].copy()

# Obter a data de corte
data_corte = df_oot['Timestamp'].min()

print("="*60)
print("DIVISÃO TEMPORAL REALIZADA")
print("="*60)
print(f"\n📊 Dataset de Treino (df_treino):")
print(f"   - Total de registros: {len(df_treino):,}")
print(f"   - Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_treino['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_treino['Is Laundering'].sum() / len(df_treino) * 100):.2f}%")

print(f"\n📊 Dataset Out-of-Time (df_oot):")
print(f"   - Total de registros: {len(df_oot):,}")
print(f"   - Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_oot['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_oot['Is Laundering'].sum() / len(df_oot) * 100):.2f}%")

print(f"\n📅 Data de corte: {data_corte}")
print(f"\n✅ Proporção treino/OOT: {split_ratio*100:.0f}% / {(1-split_ratio)*100:.0f}%")

DIVISÃO TEMPORAL REALIZADA

📊 Dataset de Treino (df_treino):
   - Total de registros: 25,001,186
   - Período: 2022-09-01 00:00:00 até 2022-09-14 05:39:00
   - Casos de lavagem: 12,258
   - Taxa de lavagem: 0.05%

📊 Dataset Out-of-Time (df_oot):
   - Total de registros: 6,250,297
   - Período: 2022-09-14 05:39:00 até 2022-09-27 14:58:00
   - Casos de lavagem: 3,783
   - Taxa de lavagem: 0.06%

📅 Data de corte: 2022-09-14 05:39:00

✅ Proporção treino/OOT: 80% / 20%


## 6. Salvamento dos Datasets

In [6]:
# Definir os caminhos de saída
output_dir = r'C:\Users\win\Desktop\TCC\money_laundering\data\processed'
os.makedirs(output_dir, exist_ok=True)

# Caminhos dos arquivos de saída
treino_path = os.path.join(output_dir, 'df_treino.csv')
oot_path = os.path.join(output_dir, 'df_oot.csv')

# Remover a coluna Year_Month antes de salvar (coluna auxiliar)
df_treino_save = df_treino.drop(columns=['Year_Month'], errors='ignore')
df_oot_save = df_oot.drop(columns=['Year_Month'], errors='ignore')

# Salvar os datasets
print("Salvando datasets...")
df_treino_save.to_csv(treino_path, index=False)
df_oot_save.to_csv(oot_path, index=False)

print(f"\n✅ Datasets salvos com sucesso!")
print(f"\n📁 Arquivos salvos em:")
print(f"   - Treino: {treino_path}")
print(f"   - OOT: {oot_path}")

# Verificar tamanhos dos arquivos
treino_size = os.path.getsize(treino_path) / (1024 * 1024)  # MB
oot_size = os.path.getsize(oot_path) / (1024 * 1024)  # MB

print(f"\n📏 Tamanho dos arquivos:")
print(f"   - Treino: {treino_size:.2f} MB")
print(f"   - OOT: {oot_size:.2f} MB")

Salvando datasets...

✅ Datasets salvos com sucesso!

📁 Arquivos salvos em:
   - Treino: C:\Users\win\Desktop\TCC\money_laundering\data\processed\df_treino.csv
   - OOT: C:\Users\win\Desktop\TCC\money_laundering\data\processed\df_oot.csv

📏 Tamanho dos arquivos:
   - Treino: 5608.33 MB
   - OOT: 1400.28 MB


## 7. Verificação Final

Vamos verificar a consistência dos dados salvos.

In [7]:
# Recarregar os arquivos para verificação
df_treino_verificacao = pd.read_csv(treino_path)
df_oot_verificacao = pd.read_csv(oot_path)

print("="*60)
print("VERIFICAÇÃO DOS ARQUIVOS SALVOS")
print("="*60)

print(f"\n✓ Dataset de Treino carregado:")
print(f"  - Registros: {len(df_treino_verificacao):,}")
print(f"  - Colunas: {len(df_treino_verificacao.columns)}")
print(f"  - Formato: {df_treino_verificacao.shape}")

print(f"\n✓ Dataset Out-of-Time carregado:")
print(f"  - Registros: {len(df_oot_verificacao):,}")
print(f"  - Colunas: {len(df_oot_verificacao.columns)}")
print(f"  - Formato: {df_oot_verificacao.shape}")

print(f"\n✓ Total de registros: {len(df_treino_verificacao) + len(df_oot_verificacao):,}")
print(f"✓ Registros originais: {len(df):,}")
print(f"✓ Diferença: {len(df) - (len(df_treino_verificacao) + len(df_oot_verificacao))}")

print("\n" + "="*60)
print("✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!")
print("="*60)

VERIFICAÇÃO DOS ARQUIVOS SALVOS

✓ Dataset de Treino carregado:
  - Registros: 25,001,186
  - Colunas: 21
  - Formato: (25001186, 21)

✓ Dataset Out-of-Time carregado:
  - Registros: 6,250,297
  - Colunas: 21
  - Formato: (6250297, 21)

✓ Total de registros: 31,251,483
✓ Registros originais: 31,251,483
✓ Diferença: 0

✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!
